# Radical-Aligned Structure — One-Shot Direct Run (no Drive)

Free Colab T4. Hit **Run All** and walk away. Total ~2h 20m.

There are TWO automatic browser downloads:
- after cell 12 (variance decomposition done): a partial zip safety-net
- after cell 16 (everything done): the final zip

So even if the runtime dies after cell 12 you still have ~80% of the paper.


In [ ]:
# 1. Clone repo
!rm -rf /content/repo
!git clone https://github.com/aryan35790jp/chinese_llm.git /content/repo
%cd /content/repo


In [ ]:
# 2. Install deps. The dependency-resolver warnings about numpy<2 are SAFE to ignore.
!pip install -q transformers==4.44.2 tokenizers==0.19.1 \
    numpy==1.26.4 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.1 \
    matplotlib==3.9.0 tqdm==4.66.4 datasets==2.20.0 sentencepiece==0.2.0 \
    Pillow==10.4.0 umap-learn==0.5.6 \
    fugashi==1.3.2 unidic-lite==1.0.8 ipadic==1.0.0


In [ ]:
# 3. Confirm GPU
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# 4. Build dataset (Unihan + tokenizer filter, ~3 min)
!mkdir -p data/unihan
!curl -sL -o data/unihan/Unihan.zip https://www.unicode.org/Public/UCD/latest/ucd/Unihan.zip
!cd data/unihan && unzip -o -q Unihan.zip && cd ../..
!python scripts/new/dataset_builder.py
!python scripts/new/classify_liushu.py


In [ ]:
# 5. Tokenization audit (1 min)
import os
os.environ['RADICAL_FAST'] = '1'
!python scripts/new/tokenization_audit.py


In [ ]:
# 6. THE HEAVY STEP — extract embeddings for 10 models (~50 min on T4).
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/extract_embeddings.py


In [ ]:
# 7. Pure-vision baseline (~3 min)
!apt-get install -y -q fonts-noto-cjk 2>/dev/null || true
!python scripts/new/glyph_only_baseline.py


In [ ]:
# 8. Isotropy correction (~3 min)
!python scripts/new/isotropy_correction.py


In [ ]:
# 9. Layer-wise cohesion (~7 min, 10 models)
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/layer_wise_analysis.py


In [ ]:
# 10. Pseudoradical specificity test (~18 min with B=100).
import os
os.environ['RADICAL_FAST'] = '1'
os.environ['RADICAL_PSEUDO_B'] = '100'
!RADICAL_FAST=1 RADICAL_PSEUDO_B=100 python scripts/new/pseudoradical_control.py


In [ ]:
# 11. Frequency-matched + random-init + semantic + phon-vs-sem + cross-script + glyph-comp + scaling + orth-arith + downstream (~7 min combined)
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/frequency_matched_pairs.py
!RADICAL_FAST=1 python scripts/new/random_init_baseline.py
!RADICAL_FAST=1 python scripts/new/expanded_semantic_control.py
!RADICAL_FAST=1 python scripts/new/phonetic_vs_semantic_radicals.py
!RADICAL_FAST=1 python scripts/new/cross_script_japanese.py
!RADICAL_FAST=1 python scripts/new/glyph_comparison.py
!RADICAL_FAST=1 python scripts/new/scaling_analysis.py
!RADICAL_FAST=1 python scripts/new/orthographic_arithmetic.py
!RADICAL_FAST=1 python scripts/new/downstream_validation.py


In [ ]:
# 12. Variance decomposition (~25 min — Wikipedia stream first time, then cached)
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/cooccurrence_baseline.py

# >>> SAFETY-NET DOWNLOAD <<<
# At this point we have everything except cluster-robust SE, non-parametric stats,
# and the cloze probe. Pack and download a partial zip so even if the runtime
# dies in the next ~30 min, you've banked ~80% of the paper.
!python scripts/new/figures.py 2>/dev/null || true
!python scripts/new/results_report.py 2>/dev/null || true
!cd /content/repo && zip -qr /content/results_partial.zip results figures data/radical_dataset.csv data/radical_summary.csv
!ls -lh /content/results_partial.zip
from google.colab import files
files.download('/content/results_partial.zip')
print('\n[safety-net] partial zip downloaded — feel free to keep going')


In [ ]:
# 13. Cluster-robust SE regression (~3 min) + non-parametric stats (~3 min)
!python scripts/new/cluster_robust_regression.py
!python scripts/new/nonparametric_stats.py


In [ ]:
# 14. Procedural cloze items + probe (~25 min for 7 models, the behavioural evidence)
import pathlib
for f in ['radical_cloze.csv', 'radical_cloze_summary.csv']:
    p = pathlib.Path('results') / f
    if p.exists(): p.unlink()
!python scripts/new/cloze_procedural.py
!python scripts/new/radical_cloze_probe.py


In [ ]:
# 15. Figures + auto report
!python scripts/new/figures.py
!python scripts/new/results_report.py

with open('results/_REPORT.md', encoding='utf-8') as f:
    from IPython.display import Markdown
    md_text = f.read()
Markdown(md_text)


In [ ]:
# 16. FINAL PACK AND DOWNLOAD
!cd /content/repo && zip -qr /content/results_v3.zip results figures data/radical_dataset.csv data/radical_summary.csv
!ls -lh /content/results_v3.zip
from google.colab import files
files.download('/content/results_v3.zip')
print('\n=== ALL DONE — final zip downloading to your browser ===')
